# Tokenization
Tokenization is the process of breaking down text into smaller units, such as words, phrases, or symbols, which are called tokens. It is a fundamental step in natural language processing (NLP) and text analysis. Tokenization is important because it enables computers to process and analyze text data by converting it into a structured format that algorithms can understand. This step is crucial for tasks like text classification, sentiment analysis, and machine translation.

# Trie (Prefix Tree)
A Trie, also known as a Prefix Tree, is a tree-like data structure that stores strings in a way that facilitates efficient retrieval, insertion, and search operations. Each node in the Trie represents a single character, and paths from the root to a leaf node represent complete strings. Tries are particularly useful for tasks that involve prefix-based queries, such as autocomplete, spell checking, and dictionary lookups.

In natural language processing (NLP), Tries are commonly used because they allow for fast prefix matching, which is essential for tokenization, searching for words in a corpus, and implementing algorithms like n-gram models. Their hierarchical structure also makes them memory-efficient for storing large vocabularies, as common prefixes are shared among words.

Let's implement a Trie first, which will be used in the tokenization step later.

In [28]:
class TrieNode:
    """Representation of a TrieNode"""
    def __init__(self, char: str, is_end: bool = False):
        self.char = char
        self.children = {}
        self.is_end = is_end

    def __str__(self) -> str:
        return f"TrieNode(is_end={self.is_end}, children={list(self.children.keys())})"

    def __repr__(self) -> str:
        return self.__str__()

class Trie:
    """Simple implementation of Trie (Prefix Tree)"""
    def __init__(self):
        self.root = TrieNode("<root>")

    def add(self, word: str):
        def _add(node: TrieNode, rest_word: str):
            if len(rest_word) == 0:
                node.is_end = True
                return
            matched_node = node.children.get(rest_word[0])
            if matched_node:
                _add(matched_node, rest_word[1:])
            else:
                node.children[rest_word[0]] = TrieNode(rest_word[0])
                _add(node.children[rest_word[0]], rest_word[1:])
        _add(self.root, word)
    
    def find_longest_prefix(self, text: str, start_index: int = 0) -> str | None:
        longest_token = None
        def _find_lonest_prefix(node: TrieNode, rest_word: str, token: str):
            nonlocal longest_token
            if node.is_end:
                if longest_token is None or len(token) > len(longest_token):
                    longest_token = token
            if len(rest_word) == 0:
                return longest_token
            matched_node = node.children.get(rest_word[0])
            if matched_node:
                _find_lonest_prefix(matched_node, rest_word[1:], token + rest_word[0])

        _find_lonest_prefix(self.root, text[start_index:], "")
        return longest_token
    
    def inspect(self) -> str:
        """
        Visualize the Trie structure as a string.
        """
        def _inspect(node: TrieNode, prefix: str, depth: int) -> str:
            result = "  " * depth + f"{node.char} (End: {node.is_end})\n"
            for child in node.children.values():
                result += _inspect(child, prefix + child.char, depth + 1)
            return result

        return _inspect(self.root, "", 0)

In [29]:
trie = Trie()
trie.add("hello")
trie.add("world")
trie.add("hi")
print(trie.inspect())

<root> (End: False)
  h (End: False)
    e (End: False)
      l (End: False)
        l (End: False)
          o (End: True)
    i (End: True)
  w (End: False)
    o (End: False)
      r (End: False)
        l (End: False)
          d (End: True)



In [30]:
trie.find_longest_prefix("worldy")

'world'

As you can see above, "world" was added to the Trie and the path "w-o-r-l-d" ends with the character "d". Therefore, when we look up the longest prefix for "wordly" from the tree, "world" is returned.

Before implementing the tokenizer, let's first write a helper function to preprocess text. Preprocessing is essential because it ensures that the input text is in a consistent format, which simplifies subsequent tokenization and analysis.

In [31]:
import re

def preprocess_text(text: str) -> str:
    """
    Preprocesses the input text by converting it to lowercase and removing extra whitespace.
    This ensures the text is in a consistent format for tokenization.
    """
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [32]:
preprocess_text("Hello World!  ")

'hello world!'

# BPE (Byte Pair Encoding) Tokenization
Byte Pair Encoding (BPE) is a subword tokenization technique that iteratively merges the most frequent pairs of characters or character sequences in a corpus. It is commonly used in transformer training because it strikes a balance between character-level and word-level tokenization, allowing the model to handle rare and out-of-vocabulary words effectively. BPE reduces the vocabulary size while preserving the ability to represent complex words, making it computationally efficient and suitable for multilingual and large-scale datasets.

In [33]:
class BytePairEncodingTokenizer:
    """A simple implementation of a Byte Pair Encoding (BPE) tokenizer."""
    def __init__(self, target_vocab_size=25):
        self.vocab = set()
        self.target_vocab_size = target_vocab_size

    def train(self, texts: list[str]):
        # First scan just to construct the initial vocab without merging tokens
        for text in texts:
            self.vocab.update([c for c in preprocess_text(text)])
        # The main loop to progressively perform merging of most frequent adjacent tokens
        while len(self.vocab) < self.target_vocab_size:
            frequency = {}
            for text in texts:
                tokens = self.tokenize(text)
                freq = self.adjacent_tokens_frequency(tokens)
                for pair in freq:
                    if pair not in frequency:
                        frequency[pair] = freq[pair]
                    else:
                        frequency[pair] += freq[pair]
            tok1, tok2 = max(frequency, key=frequency.get)
            self.vocab.add(tok1 + tok2)

    def adjacent_tokens_frequency(self, tokens: list [str]) -> dict[tuple[str, str], int]:
        frequency = {}
        for index in range(len(tokens) - 1):
            tok1 = tokens[index]
            tok2 = tokens[index + 1]
            if (tok1, tok2) not in frequency:
                frequency[(tok1, tok2)] = 0
            frequency[(tok1, tok2)] += 1
        return frequency

    def tokenize(self, text: str, vocab: set[str] = None, unknown_token: str="<unk>") -> list[str]:
        text = preprocess_text(text)
        vocab = vocab or self.vocab
        trie = Trie()
        for token in vocab:
            trie.add(token)
        index = 0
        tokens = []
        while index < len(text):
            longest_token = trie.find_longest_prefix(text, index)
            if longest_token:
                tokens.append(longest_token)
                index += len(longest_token)
            else:
                tokens.append(unknown_token)
                index += 1
        return tokens

    def tokenize_navie(self, text: str, vocab: set[str] = None, unknown_token: str = "<unk>") -> list[str]:
        text = preprocess_text(text)
        vocab = vocab or self.vocab
        sorted_vocab = sorted(vocab, key=len, reverse=True)
        tokens = []
        index = 0
        while index < len(text):
            char = text[index]
            candidates = [token for token in sorted_vocab if token.startswith(char)]
            for candidate in candidates:
                if text[index : index + len(candidate)] == candidate:
                    tokens.append(candidate)
                    index += len(candidate)
                    break
            else:
                tokens.append(unknown_token)
                index += 1
        return tokens

Let's try to train the tokenizer with a small vocab size and data.

In [34]:
tokenizer = BytePairEncodingTokenizer(target_vocab_size=30)
tokenizer.train([
    "hello world",
    "jello tastes good",
    "say hello to you",
])
tokenizer.vocab

{' ',
 'a',
 'd',
 'e',
 'el',
 'ell',
 'ello ',
 'g',
 'h',
 'hello ',
 'hello w',
 'hello wo',
 'hello wor',
 'hello worl',
 'hello world',
 'j',
 'jello ',
 'jello t',
 'jello ta',
 'jello tas',
 'jello tast',
 'l',
 'o',
 'o ',
 'r',
 's',
 't',
 'u',
 'w',
 'y'}

As we can see, becuase the word "hello" appears frequent enough, our tokenizer now learns it.

Let's train the tokenizer with a small amount of data, since it is not very efficient. We will use the first 10000 lines of the `tiny_shakespeare` sample data, with a `target_vocab_size=500`.

In [35]:
tokenizer = BytePairEncodingTokenizer(target_vocab_size=500)
lines_limit = 10000  # Read only portion of the text

with open('./data/tiny_shakespeare.txt') as f:
    texts = []
    for line in f.readlines(lines_limit):
        strip_line = line.strip()
        if strip_line:
            texts.append(preprocess_text(strip_line))

tokenizer.train(texts)
tokenizer.tokenize("see what she becomes")

['se', 'e ', 'what ', 'sh', 'e ', 'be', 'com', 'es']

# Note on Implementation
The implementation of the Byte Pair Encoding (BPE) tokenizer provided in this notebook is a simplified version designed for educational purposes. It lacks the optimizations and robustness of production-grade tokenizers. When implementing transformers or other NLP models in real-world applications, it is highly recommended to use well-trained and widely adopted tokenizers, such as those provided by libraries like Hugging Face's `transformers` or `sentencepiece`. These libraries offer efficient, scalable, and pre-trained tokenizers that are optimized for various tasks and datasets.